In [2]:
%pip install ase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 5.0 MB/s eta 0:00:00a 0:00:01m
Note: you may need to restart the kernel to use updated packages.


In [3]:
import ase

In [ ]:
from dataclasses import dataclass
from typing import Dict, Optional

@dataclass
class Site:
    frac: np.ndarray #shape (3,)
    species: Dict[str,float]
    occ: float = 1.0
    label: Optional[str] = None

@dataclass
class ConventionalCell():
    a: float
    b: float
    c: float
    alpha: float
    beta: float
    gamma: float
    basis: list[Site] #fractional coordinates


import numpy as np
@dataclass
class Lattice():
    a1: np.ndarray
    a2: np.ndarray
    a3: np.ndarray

    @staticmethod
    def from_conventional_cell_param(a,b,c,alpha,beta,gamma,angle="degrees") -> "Lattice":
        if angle=="degrees":
          alpha = np.radians(alpha)
          beta = np.radians(beta)
          gamma = np.radians(gamma)

        a1 = np.ndarray([a,0.0,0.0])
        a2 = np.ndarray([
            b * np.cos(gamma),
            b * np.sin(gamma),
            0.0
        ])
        
        cx = c * np.cos(beta)
        cy = c * (np.cos(alpha) - np.cos(beta) * np.cos(gamma)) / np.sin(gamma)
        cz2 = c**2 - cx**2 - cy**2
        if cz2 < 0 and abs(cz2) < 1e-12:
            cz2 = 0.0
        if cz2 < 0:
            raise ValueError("Invalid cell parameters: cannot construct real a3 (cz^2 < 0).")
        cz = np.sqrt(cz2)

        a3 = np.array([cx, cy, cz], dtype=float)

        return Lattice(a1=a1, a2=a2, a3=a3)

    @staticmethod
    def from_conventional_cell(cell:ConventionalCell) -> "Lattice":
        return Lattice.from_conventional_cell_param(
            a=cell.a, 
            b=cell.b, 
            c=cell.c,
            alpha=cell.alpha,
            beta=cell.beta, 
            gamma=cell.gamma
        )
    
    @property
    def A(self) -> np.ndarray:
        return np.column_stack([self.a1, self.a2, self.a3])
    
    def fractional_to_cartesian(self, r: np.ndarray) -> np.ndarray:
        r = np.asarray(r, dtype=float).reshape(3,)
        return self.A @ r
    
    def cartesian_to_fractional(self, x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype=float).reshape(3,)
        if x.shape != (3,0):
            raise ValueError(f"Cartesian coordinate must have shape (3,), got {x.shape}")
        return np.linalg(self.A, x)


Si_conventionalcell = ConventionalCell(
    a=5.44, #angs
    b=5.44, #angs
    c=5.44, #angs
    alpha=90., #degs
    beta=90., #degs
    gamma=90. #degs
)




